# 技能4 · Day 3 上机：Agent经济 + 新兴商业模式

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **mesa** 构建Agent经济仿真--消费者Agent/商家Agent/AI中介Agent三类主体交互，涌现市场价格/财富分布/存活率
2. 理解Agent经济三层模型（Agent-as-Tool / Agent-as-Worker / Agent-as-Actor）的计算化建模方法
3. 用 **pandas + matplotlib** 分析仿真涌现结果--基尼系数/价格分布/Agent存活率/A2A交易量
4. 理解推理成本对Agent经济行为的约束，以及平台抽成（30%真实比例）对生态的影响
5. 建立天道推演×多Agent仿真的同构认知--仿真本质是计算化的天道推演沙盘

## 真实库与真实数据
- **mesa**（agent-based modeling 框架）：构建Agent经济仿真
- **pandas + matplotlib**：仿真结果分析与可视化
- **numpy**：仿真数学计算
- **真实经济参数**：平台抽成30%（Apple/Amazon真实比例）、Token定价$5/1M（GPT-4o真实定价）、推理成本约束

> 详见 data/README.md


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 所有库（mesa/pandas/matplotlib/numpy）均为本地可用库，不需要API Key。


In [ ]:
# !pip install mesa pandas matplotlib numpy -q

import warnings
warnings.filterwarnings('ignore')

import mesa
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # 非交互式后端
import matplotlib.pyplot as plt
from mesa.datacollection import DataCollector

print(f"mesa {mesa.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print("Agent经济仿真环境就绪")


## 1. 真实经济参数

Agent经济仿真的参数基于真实世界的经济数据，不是编造的数字：

| 参数 | 值 | 真实来源 |
|------|-----|---------|
| 平台抽成率 | 30% | Apple App Store / Amazon Marketplace 真实抽成比例 |
| Token定价 | $5/1M tokens | GPT-4o input 定价（OpenAI 2024-2025定价页） |
| 每次匹配推理token | 500 tokens | Agent协商/比价/决策的合理token消耗 |
| 推理成本/匹配 | ~$0.0025 | 500 tokens × $5/1M |

**推理成本是Agent经济的核心约束**--AI中介每次匹配都消耗token，这直接影响其经济可行性。


In [ ]:
# 真实经济参数（已定义，直接运行）
PLATFORM_COMMISSION_RATE = 0.30  # Apple/Amazon真实平台抽成
TOKEN_PRICE_PER_1M = 5.0         # GPT-4o input定价
TOKENS_PER_MATCH = 500           # 每次匹配推理token
REASONING_COST_PER_MATCH = (TOKENS_PER_MATCH / 1_000_000) * TOKEN_PRICE_PER_1M

print(f"平台抽成率: {PLATFORM_COMMISSION_RATE*100:.0f}%")
print(f"推理成本/匹配: ${REASONING_COST_PER_MATCH:.4f}")


## 2. TODO 1：消费者Agent

**消费者Agent** 是Agent经济中的需求方。每个消费者有：
- `wealth`：预算（初始1000）
- `demand`：需求量
- `alive`：是否存活（预算耗尽则破产）
- `purchases`：累计购买次数

**行为逻辑**：
1. 如果有AI中介可用，通过中介寻找最低价商家购买（支付 中介费）
2. 如果无中介，随机选择商家直接购买
3. 预算耗尽（< $1）则破产

**营销映射**：消费者Agent代客比价是Agent经济在营销中的典型实例。


In [ ]:
# TODO 1：消费者Agent
# 提示：继承mesa.Agent，实现step()方法
#   属性：wealth=1000, demand=1, alive=True, purchases=0
#   行为：通过AI中介找最低价 / 直接随机购买；预算<1则破产

class ConsumerAgent(mesa.Agent):
    # ===== 你的代码 =====

    # raise NotImplementedError


## 3. TODO 2：商家Agent

**商家Agent** 是Agent经济中的供给方。每个商家有：
- `wealth`：资金（初始500）
- `base_cost`：生产成本
- `price`：当前售价（动态调整）
- `inventory`：库存
- `commission_paid`：累计支付平台抽成

**行为逻辑**：
1. 通过中介或直接销售，每次销售支付30%平台抽成
2. 库存高则降价，库存低则涨价（动态定价）
3. 定期补货（消耗资金）
4. 资金为负则破产

**真实参数**：平台抽成30%来自Apple App Store / Amazon Marketplace的真实比例。


In [ ]:
# TODO 2：商家Agent
# 提示：继承mesa.Agent
#   属性：wealth=500, base_cost=10, price=20, inventory=100, commission_paid=0
#   方法：sell_via_intermediary(int) / sell_direct() -> 支付30%抽成 + 动态调价
#   行为：补货 + 破产检查

class MerchantAgent(mesa.Agent):
    # ===== 你的代码 =====

    # raise NotImplementedError


## 4. TODO 3：AI中介Agent

**AI中介Agent** 是Agent经济的核心创新--它用AI能力匹配供需，每次匹配消耗推理token。

核心属性：
- `wealth`：资金（初始200）
- `fee`：中介费（动态调整）
- `transactions`：累计匹配次数
- `a2a_trades`：A2A交易次数（与其他中介的交易）
- `total_reasoning_cost`：累计推理成本

**行为逻辑**：
1. 每次匹配收取fee，支付推理成本（500 tokens × $5/1M = $0.0025）
2. 以15%概率与其他中介进行A2A信息交换（支付小额费用）
3. 根据竞争者平均费率动态调整自己的fee
4. 资金为负则破产

**A2A经济**：Agent-to-Agent交易是Agent经济最前沿的形态--Agent间自主协商、交易。


In [ ]:
# TODO 3：AI中介Agent
# 提示：继承mesa.Agent
#   属性：wealth=200, fee=2.0, transactions=0, a2a_trades=0, total_reasoning_cost=0
#   方法：process_transaction() -> 收fee, 支付推理成本
#   行为：A2A交易(15%概率) + 动态调费 + 破产检查

class AIIntermediaryAgent(mesa.Agent):
    # ===== 你的代码 =====

    # raise NotImplementedError


## 5. TODO 4：Agent经济模型 + DataCollector

**AgentEconomyModel** 整合三类Agent，用mesa的DataCollector追踪涌现指标：

| 指标 | 含义 |
|------|------|
| `gini` | 基尼系数（财富不平等程度） |
| `avg_price` | 市场平均价格 |
| `price_std` | 价格标准差（价格收敛程度） |
| `n_alive_*` | 各类Agent存活数 |
| `total_a2a_trades` | 累计A2A交易量 |
| `total_commission` | 累计平台抽成 |

**天道推演映射**：模型每一步step()就是一次沙盘推演--观察不同Agent策略下的经济涌现。


In [ ]:
# TODO 4：Agent经济模型 + DataCollector
# 提示：继承mesa.Model
#   __init__: 创建三类Agent + DataCollector(8个model_reporters + 3个agent_reporters)
#   _compute_gini: 基尼系数公式 G = 2*sum((i+1)*w)/(n*sum(w)) - (n+1)/n
#   step: agents.shuffle_do("step") + datacollector.collect

class AgentEconomyModel(mesa.Model):
    # ===== 你的代码 =====

    # raise NotImplementedError


## 6. TODO 5：运行仿真 + 提取数据

运行Agent经济仿真100个tick，用DataCollector提取时间序列数据到pandas DataFrame。

**关键问题**：
- 基尼系数如何变化？（财富是否越来越集中？）
- 市场价格是否收敛？
- 哪类Agent最先破产？
- A2A交易量增长趋势如何？


In [ ]:
# TODO 5：运行仿真 + 提取数据
# 提示：运行100步，用datacollector提取model_vars和agent_vars到DataFrame
#   打印仿真规模、最终基尼、价格分布、存活数、A2A交易量

# ===== 你的代码 =====

# raise NotImplementedError


## 7. TODO 6：仿真结果分析与可视化

用pandas分析仿真涌现结果，用matplotlib绘制4个子图：
1. 基尼系数随时间变化（财富不平等趋势）
2. 市场价格分布（均值±标准差）
3. Agent存活数（消费者/商家/中介三条线）
4. A2A交易 vs 中介交易量对比

**涌现分析**：这些指标是Agent个体行为的涌现结果--没有任何单个Agent"知道"全局价格或基尼系数，它们在交互中自然产生。


In [ ]:
# TODO 6：仿真结果分析与可视化
# 提示：用matplotlib绘制4个子图
#   1. 基尼系数随时间变化 2. 市场价格分布(均值±std)
#   3. Agent存活数(3条线) 4. A2A vs 中介交易量

# ===== 你的代码 =====

# raise NotImplementedError


## 8. 天道推演 × 多Agent仿真

本仿真本质是**计算化的天道推演沙盘**：

| 天道推演能力 | 仿真对应 | 涌现产出 |
|-------------|---------|---------|
| 局势感知 | 初始Agent分布与参数 | 初始基尼/价格 |
| 因果链追踪 | Agent行为因果（购买->降价->竞争） | 价格动态 |
| 沙盘模拟（3层） | 100 tick推演 | 时间序列涌现 |
| 概率评估 | 多次运行不同seed | 结果分布 |
| 最优路径推荐 | 对比不同参数场景 | 策略选择 |

**核心洞察**：Agent经济仿真让天道推演从"意识中的沙盘"变为"可计算、可复现的沙盘"。

## 交付物
- [ ] 完成的 starter.ipynb（6个TODO全部填好）
- [ ] 4个子图的仿真结果可视化
- [ ] 一段300字分析：仿真涌现了什么经济现象？推理成本对AI中介的影响？
